# UniAssist — Week 2: Tokenization, Sentence Boundaries and Subwords
Compare whitespace, NLTK and spaCy on the original corpus. Complete manual tokenization before judging which tokenizer is best.


In [1]:
from pathlib import Path
import json, re, statistics
from collections import Counter
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "corpus").exists():
    ROOT = ROOT.parent
assert (ROOT / "corpus").exists(), "Run notebook from UniAssist_UMT directory"
metadata = json.loads((ROOT / "metadata.json").read_text(encoding="utf-8"))
records = metadata["documents"]
texts = [(ROOT / "corpus" / row["file"]).read_text(encoding="utf-8") for row in records]
print(len(texts), "documents loaded")


20 documents loaded


In [2]:
import nltk, spacy
from nltk.tokenize import word_tokenize, sent_tokenize
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")
methods = {
    "split": lambda t: t.split(),
    "nltk": lambda t: word_tokenize(t, preserve_line=True),
    "spaCy": lambda t: [w.text for w in nlp.make_doc(t) if not w.is_space],
}


## Task A — Token/type counts and type-token ratios


In [3]:
rows = []
for name, fn in methods.items():
    toks = [word for text in texts for word in fn(text)]
    types = len(set(w.lower() for w in toks))
    punct = sum(not any(ch.isalnum() for ch in word) for word in toks)
    rows.append({"Tokenizer": name, "Tokens": len(toks), "Types": types,
                 "Types/Tokens": round(types / max(1, len(toks)), 4),
                 "Punct. tokens": punct})
display(pd.DataFrame(rows))


,Tokenizer,Tokens,Types,Types/Tokens,Punct. tokens
0,split,5068,1555,0.3068,0
1,nltk,5708,1318,0.2309,541
2,spaCy,6073,1173,0.1932,854


## Task B — Regex problem hunt: URLs, emails, dates and punctuation


In [4]:
patterns = {
    "url": r"https?://\S+|www\.\S+",
    "email": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "number": r"\d[\d.,:/-]*\d|\d",
    "hyphenated": r"\w+(?:-\w+)+",
    "apostrophe": r"\w+['’]\w+",
    "abbreviation": r"\b[A-Z][a-z]{0,3}\.",
    "other_symbol": r"[^\w\s.,;:!?()'\-]"
}
found_rows = []
for kind, pattern in patterns.items():
    matches = [(record["file"], m.group()) for record, text in zip(records, texts)
               for m in re.finditer(pattern, text)]
    found_rows.append({"type": kind, "number": len(matches), "three_examples": matches[:3]})
display(pd.DataFrame(found_rows))


,type,number,three_examples
0,url,20,"[(01_university_overview.txt, https://umt.edu...."
1,email,0,[]
2,number,37,"[(06_bachelor_admissions_faq.txt, 295), (06_ba..."
3,hyphenated,91,"[(01_university_overview.txt, higher-education..."
4,apostrophe,68,"[(01_university_overview.txt, bachelor's), (01..."
5,abbreviation,5,"[(06_bachelor_admissions_faq.txt, No.), (06_ba..."
6,other_symbol,267,"[(01_university_overview.txt, /), (01_universi..."


## Task C — Compare the same five tricky expressions across tokenizers


In [5]:
hard = ["2026–2027", "Erasmus+", "Engineer-Architect",
        "State Matura", "https://umt.edu.al/en/admissions/"]
for sample in hard:
    print("\nEXPRESSION:", sample)
    for name, tokenize in methods.items():
        print(name, len(tokenize(sample)), tokenize(sample))



EXPRESSION: 2026–2027
split 1 ['2026–2027']
nltk 3 ['2026', '–', '2027']
spaCy 1 ['2026–2027']

EXPRESSION: Erasmus+
split 1 ['Erasmus+']
nltk 1 ['Erasmus+']
spaCy 1 ['Erasmus+']

EXPRESSION: Engineer-Architect
split 1 ['Engineer-Architect']
nltk 1 ['Engineer-Architect']
spaCy 3 ['Engineer', '-', 'Architect']

EXPRESSION: State Matura
split 2 ['State', 'Matura']
nltk 2 ['State', 'Matura']
spaCy 2 ['State', 'Matura']

EXPRESSION: https://umt.edu.al/en/admissions/
split 1 ['https://umt.edu.al/en/admissions/']
nltk 3 ['https', ':', '//umt.edu.al/en/admissions/']
spaCy 1 ['https://umt.edu.al/en/admissions/']


## Task D — Sentence segmentation and mean/median sentence lengths


In [6]:
def spacy_sents(text):
    return [s.text.strip() for s in nlp(text).sents if s.text.strip()]

sentences = [s for text in texts for s in spacy_sents(text)]
lengths = [(len(methods["spaCy"](s)), s) for s in sentences]
nums = [n for n,_ in lengths]
print("Sentences", len(sentences), "Mean", statistics.mean(nums), "Median", statistics.median(nums))
for count, sentence in sorted(lengths, reverse=True)[:5]:
    print(count, sentence[:170])
sample = texts[14]
print("Comparing NLTK and spaCy for one document:")
print("NLTK", len(sent_tokenize(sample)), "spaCy", len(spacy_sents(sample)))


Sentences 317 Mean 19.157728706624606 Median 19
48 The faculty's postgraduate list includes a Professional Master in Digital Marketing, Master of Economics (MBA), Master of Science in Management Engineering, Master of Sci
45 Search queries about the "AI lab," "software lab," "campus buildings" or "Qyteti Studenti" should find this record, but the answer must stay within the university's publi
44 The distinction between the language of instruction and the country in which the university is located is important: students interested in studying in English should sea
40 Grading and Examination Records in a Published UMT Regulation

The publicly available UMT Examination Regulations PDF describes a numerical grading system running from 4 
39 Terms such as "2026–2027," "Academic Senate," "PDF" and "timetable" are useful tokenization examples, but preservation of the original source is more important than produ
Comparing NLTK and spaCy for one document:
NLTK 16 spaCy 16


## Optional Task E — Subword comparison
If you install transformers and sentencepiece, the following cell downloads model tokenizers. It may require internet access.


In [7]:
try:
    from transformers import AutoTokenizer
except ImportError:
    print("Optional dependency not installed: pip install transformers sentencepiece")
else:
    models = {"BERT":"bert-base-uncased", "GPT-2":"gpt2",
              "XLM-R":"xlm-roberta-base"}
    for name, checkpoint in models.items():
        try:
            tok = AutoTokenizer.from_pretrained(checkpoint)
            print(name, {word:tok.tokenize(word) for word in
                  ("university", "registration", "Erasmus+", "Metropolitan Tirana")})
        except Exception as err:
            print(name, "could not download:", type(err).__name__)


/private/tmp/uniassist-notebooks-01a1/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


BERT {'university': ['university'], 'registration': ['registration'], 'Erasmus+': ['erasmus', '+'], 'Metropolitan Tirana': ['metropolitan', 'tirana']}


GPT-2 {'university': ['un', 'iversity'], 'registration': ['reg', 'istration'], 'Erasmus+': ['Er', 'asm', 'us', '+'], 'Metropolitan Tirana': ['Met', 'ropolitan', 'ĠTir', 'ana']}


XLM-R {'university': ['▁university'], 'registration': ['▁registration'], 'Erasmus+': ['▁Erasmus', '+'], 'Metropolitan Tirana': ['▁Metropolitan', '▁Tirana']}


## Submit
Show the three tokenization tables, five difficult strings, mean/median sentence lengths, five longest sentences and your explanations. Explicitly distinguish manual decisions from software output.
